<a href="https://colab.research.google.com/github/broadinstitute/BE3D/blob/main/examples/BE3Dv9_MultiScreen_KBTBD4_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# README
https://github.com/broadinstitute/BE3D

# Setup

In [ ]:
# @title Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')
from google.colab import output
output.enable_custom_widget_manager()


In [ ]:
# @title Install DSSP and ClustalO

! apt-get update
! apt-get install dssp clustalo


In [ ]:
# @title Install MUSCLE

! wget https://github.com/rcedgar/muscle/releases/download/v5.3/muscle-linux-x86.v5.3
! chmod +x muscle-linux-x86.v5.3
! mv muscle-linux-x86.v5.3 muscle


In [ ]:
# @title Install BE3D

! pip install git+https://github.com/broadinstitute/beclust3d-public.git
print('beclust3d installed at:')
! pip show beclust3d


In [ ]:
# @title Import packages

import os
import sys
import yaml
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import gc
import uuid
import ipywidgets as widgets
from IPython.display import display

from IPython.display import Image, display, SVG
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from google.colab import files
import shutil

from beclust3d import *

import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

plt.rcParams['font.family'] = 'DejaVu Sans'


In [ ]:
# @title Download relevant files for running BE3D

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/be3d_display_yaml.py -O be3d_display_yaml.py
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/be3d_helper.py -O be3d_helper.py

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/be3d_qa.py -O be3d_qa.py

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/beclust3d_calculate_lfc3d.py -O beclust3d_calculate_lfc3d.py
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/beclust3d_nonaggregate.py -O beclust3d_nonaggregate.py
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/beclust3d_characterization.py -O beclust3d_characterization.py

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/bemetaclust3d_metaaggregate.py -O bemetaclust3d_metaaggregate.py
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/bemetaclust3d_characterization.py -O bemetaclust3d_characterization.py

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/be3d_config_editor.py -O be3d_config_editor.py
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/be3d_plotly.py -O be3d_plotly.py


# BE3D Inputs (KBTBD4 chain A, chain B PPI, and HDAC1 chain C)

In [ ]:
# @title Download relevant files for KBTBD4 (chain A, chain B PPI) and HDAC1 (chain C)

! mkdir KBTBD4/
! mkdir HDAC1/

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/yaml/KBTBD4_chain_A_pdb.yaml -O KBTBD4/KBTBD4_chain_A_pdb.yaml
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/yaml/KBTBD4_chain_B_pdb_PPI_heterotrimer.yaml -O KBTBD4/KBTBD4_chain_B_pdb_PPI_heterotrimer.yaml
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/yaml/HDAC1_chain_C_pdb.yaml -O HDAC1/HDAC1_chain_C_pdb.yaml

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/pdb/KBTBD4_8voj_single_chain_A.pdb -O KBTBD4/KBTBD4_8voj_single_chain_A.pdb
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/pdb/KBTBD4_HDAC1_8voj_chain_ABC.pdb -O KBTBD4/KBTBD4_HDAC1_8voj_chain_ABC.pdb
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/pdb/HDAC1_8voj_single_chain_C_w_ligands.pdb -O HDAC1/HDAC1_8voj_single_chain_C_w_ligands.pdb

# KBTBD4 and HDAC1 share the same screen_name (abe_neg_control / cbe_neg_control) on
# purpose -- calculate_lfc3d.py's PPI lookup derives the interacting gene's
# protein_edits.tsv filename from the *requesting* run's own screen names, so
# both sides of the heterotrimer need identical basenames even though the
# underlying per-gene data lives in separate subfolders.
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/data/YeoNature2025-KBTBD4-HDAC1/KBTBD4/abe_neg_control.tsv -O KBTBD4/abe_neg_control.tsv
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/data/YeoNature2025-KBTBD4-HDAC1/KBTBD4/cbe_neg_control.tsv -O KBTBD4/cbe_neg_control.tsv
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/data/YeoNature2025-KBTBD4-HDAC1/HDAC1/abe_neg_control.tsv -O HDAC1/abe_neg_control.tsv
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/data/YeoNature2025-KBTBD4-HDAC1/HDAC1/cbe_neg_control.tsv -O HDAC1/cbe_neg_control.tsv


In [ ]:
# @title Load Default Config — KBTBD4 (chain A)
# @markdown This run is a prerequisite for the chain B PPI run below and isn't
# @markdown edited interactively; its config is loaded as-is.
yaml_config_filepath_A = "KBTBD4/KBTBD4_chain_A_pdb.yaml"
with open(yaml_config_filepath_A, 'r') as file:
    input_dict_A = yaml.safe_load(file)

output_dir_A = input_dict_A['output_dir']
input_gene_A = input_dict_A['input_gene']
print(f"Loaded default config for {input_gene_A} (chain A) from {yaml_config_filepath_A}. Output directory: {output_dir_A}")


In [ ]:
# @title Load Default Config — HDAC1 (chain C)
# @markdown This run is a prerequisite for the chain B PPI run below and isn't
# @markdown edited interactively; its config is loaded as-is.
yaml_config_filepath_C = "HDAC1/HDAC1_chain_C_pdb.yaml"
with open(yaml_config_filepath_C, 'r') as file:
    input_dict_C = yaml.safe_load(file)

output_dir_C = input_dict_C['output_dir']
input_gene_C = input_dict_C['input_gene']
print(f"Loaded default config for {input_gene_C} (chain C) from {yaml_config_filepath_C}. Output directory: {output_dir_C}")


In [ ]:
# @title Edit BE3D Inputs (interactive) — KBTBD4 (chain B, PPI mode)
from be3d_config_editor import edit_yaml_config

# @markdown Path to downloaded or user-input .yaml configuration file
yaml_config_filepath = "KBTBD4/KBTBD4_chain_B_pdb_PPI_heterotrimer.yaml" # @param {type:"string"}

# @markdown Fill in / adjust the fields below (add or remove screen files in the
# @markdown **Screens** section), then click **Save config** before moving on.
editor = edit_yaml_config(yaml_config_filepath)
editor


In [ ]:
# @title Load Saved Config
# @markdown Run this cell **after** clicking "Save config" above.
with open(yaml_config_filepath, 'r') as file:
    input_dict = yaml.safe_load(file)

output_dir = input_dict['output_dir']
input_gene = input_dict['input_gene']
print(f"Loaded config for {input_gene} (chain B, PPI mode). Output directory: {output_dir}")


# BE3D Run (KBTBD4 chain A, chain B PPI, and HDAC1 chain C)

In [ ]:
# @title Prerequisite Runs (KBTBD4 chain A and HDAC1 chain C)
# @markdown The chain B PPI run below reads each interacting chain's own LFC
# @markdown data (via its screendata_sequence/*_protein_edits.tsv output), so
# @markdown both of these must run first. Chain A also gets its own
# @markdown nonaggregate step, since its per-residue scores are needed for
# @markdown the side-by-side comparison in the Results section; chain C
# @markdown (HDAC1) only needs calculate_lfc3d, since it isn't shown on its own.

! python beclust3d_calculate_lfc3d.py {yaml_config_filepath_A}
! python beclust3d_nonaggregate.py {yaml_config_filepath_A}
print(f"KBTBD4 chain A run complete: {output_dir_A}")

! python beclust3d_calculate_lfc3d.py {yaml_config_filepath_C}
print(f"HDAC1 chain C run complete: {output_dir_C}")


In [ ]:
# @title BE-Clust3D (PPI mode) — KBTBD4 chain B
# @markdown Description

! python beclust3d_calculate_lfc3d.py {yaml_config_filepath}
! python beclust3d_nonaggregate.py {yaml_config_filepath}
from beclust3d.helpers.visualization.g2p import g2p_formatted_hit_cluster

screens = input_dict['screens']
if isinstance(screens, str):
    screens = [s.strip() for s in screens.split(',')]
screen_names = [s.split('.')[0] for s in screens]
gene_list = [input_gene] * len(screen_names)

pthr = input_dict.get('pthr', {})
single_pthr = str(pthr.get('single_screen', 0.05)).split('.')[1]
multi_pthr  = str(pthr.get('multi_screen', 0.001)).split('.')[1]
conservation_run = input_dict.get('conservation', {}).get('run', False)

g2p_formatted_hit_cluster(
    output_dir, gene_list, screen_names,
    lfc_pthr=single_pthr, lfc3d_pthr=single_pthr, meta_pthr=multi_pthr,
    function_for_meta=False,
    conservation=conservation_run,
    input_gene=input_gene,
)


# Results: KBTBD4 chain A vs. chain B (PPI mode)

In [ ]:
# @title Compare LFC / LFC3D scores — KBTBD4 chain A vs. chain B (PPI mode)
# @markdown Both configs share the same screen files, so the same screen_name
# @markdown is used to pull each run's per-residue scores.

from be3d_plotly import plot_score_scatter, show_side_by_side

screen_name = 'abe_neg_control'

lfc_fig_A = plot_score_scatter(output_dir_A, input_gene_A, screen_name, score_type='LFC', pthr_str=single_pthr)
lfc_fig_B = plot_score_scatter(output_dir, input_gene, screen_name, score_type='LFC', pthr_str=single_pthr)
show_side_by_side(lfc_fig_A, lfc_fig_B, width=1000)

lfc3d_fig_A = plot_score_scatter(output_dir_A, input_gene_A, screen_name, score_type='LFC3D', pthr_str=single_pthr)
lfc3d_fig_B = plot_score_scatter(output_dir, input_gene, screen_name, score_type='LFC3D', pthr_str=single_pthr)
show_side_by_side(lfc3d_fig_A, lfc3d_fig_B, width=1000)


# G2P Visualizations

The steps below map your BE3D results onto an interactive 3D protein structure using the [Genomics 2 Proteins (G2P) portal](https://g2p.broadinstitute.org). You will download the BE3D output files, upload them to the portal, and explore base-editing scores and clusters directly on the structure.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, FileLink
import os
import shutil
from google.colab import files

def download_files_button(path):
    output_filename = f"{os.path.basename(path)}_files.zip"
    shutil.make_archive(output_filename.replace('.zip', ''), 'zip', path)

    print(f"Created {output_filename}. Click the button to download.")

    button = widgets.Button(description=f"Download {output_filename}")
    output = widgets.Output()

    def on_button_clicked(b):
        with output:
            files.download(output_filename)

    button.on_click(on_button_clicked)
    display(button, output)

In [ ]:
# @title 1. Download the G2P visualization files
# @markdown Run this cell, then click the button to download the BE3D output files prepared for G2P.
download_files_button(output_dir + 'g2p_visualization')

In [ ]:
# @title 2. Open the G2P portal interactive module
# @markdown 1. Go to https://g2p.broadinstitute.org
# @markdown 2. Click **Interactive Module**

from IPython.display import Image
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/2_g2p.png", width=600)

In [ ]:
# @title 3. Start the structure-mapping workflow
# @markdown Click **Start with a gene/protein identifier** to map BE3D results onto a 3D structure using G2P.
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/3_select_first_interactive_module.png", width=600)

In [ ]:
# @title 4. Select a gene
# @markdown Enter a gene name and click **Proceed**.
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/4_select_a_gene.png", width=600)

In [ ]:
# @title 5. Select a protein structure
# @markdown Choose a 3D protein structure from the PDB, AlphaFold, or your own uploaded structure.
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/5_select_a_pdb.png", width=600)

In [ ]:
# @title 6. Upload your BE3D results (.tsv)
# @markdown Click **Upload your data** and select the BE3D results file downloaded in step 1.
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/6_upload_BE3D_tsv.png", width=600)

In [ ]:
# @title 7. Filter the columns of interest
# @markdown 1. Click **Filter Columns** and keep the columns you want to visualize.
# @markdown 2. Set the data type to **features** for any `... hit cluster` column.

Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/7_filter_columns.png", width=600)

In [ ]:
# @title 8. Explore the visualization
# @markdown Your BE3D scores and clusters are now mapped onto the 3D structure — rotate, zoom, and explore.
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/8_done.png", width=600)